In [ ]:
#不載入用不到的
import numpy as np
import pandas as pd

from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')
import gc

In [2]:
train_features = pd.read_pickle('data/preprocessed/local_train_features.pkl')

In [3]:
train_features.head(5)

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,...,month_x,month_y,weekday_x,weekday_y,building_weekday_hour,building_weekday,building_month,building_hour,building_meter,is_holiday
2,2,0,0.0,0.0,0,Education,5376,91.0,NaN,19.40625,...,1.0,0.0,1.0,0.0,2-0-4-0,2-0-4,2-0-1,2-0-0,2-0-,1
3,3,0,0.0,0.0,0,Education,23685,102.0,NaN,19.40625,...,1.0,0.0,1.0,0.0,3-0-4-0,3-0-4,3-0-1,3-0-0,3-0-,1
5,5,0,0.0,0.0,0,Education,8000,100.0,NaN,19.40625,...,1.0,0.0,1.0,0.0,5-0-4-0,5-0-4,5-0-1,5-0-0,5-0-,1
6,6,0,0.0,0.0,0,Lodging/residential,27926,81.0,NaN,19.40625,...,1.0,0.0,1.0,0.0,6-0-4-0,6-0-4,6-0-1,6-0-0,6-0-,1
10,10,0,0.0,0.0,0,Entertainment/public assembly,370773,91.0,NaN,19.40625,...,1.0,0.0,1.0,0.0,10-0-4-0,10-0-4,10-0-1,10-0-0,10-0-,1


In [4]:
train_features["meter_reading"].isna().sum()

np.int64(0)

In [5]:
print(train_features["building_id"].dtype, train_features["timestamp"].dtype, train_features["meter_reading"].dtype)

int16 float64 float32


In [6]:
shifts = [-168,-24,-2,-1,1,2,24,168]
for shift_hours in tqdm(shifts):
    train_feature = train_features[['building_id', 'timestamp', 'meter_reading']]
    meter_reading_shift = train_features[['building_id', 'timestamp', 'meter_reading']]
    
    meter_reading_shift['timestamp'] += shift_hours
    meter_reading_shift = meter_reading_shift.rename(columns={'meter_reading':'lag_value_'+str(shift_hours)})
    train_feature = train_feature.merge(meter_reading_shift, on=['building_id', 'timestamp'], how='left')
    
    train_features['lag_value_'+str(shift_hours)] = train_feature['lag_value_'+str(shift_hours)]-train_feature['meter_reading']

100%|██████████| 8/8 [00:24<00:00,  3.09s/it]


In [7]:
for shift_hours in tqdm(shifts):
    train_feature = train_features[['building_id', 'timestamp', 'meter_reading']]
    meter_reading_shift = train_features[['building_id', 'timestamp', 'meter_reading']]
    
    meter_reading_shift['timestamp'] += shift_hours
    meter_reading_shift = meter_reading_shift.rename(columns={'meter_reading':'lag_value_'+str(shift_hours)})
    train_feature = train_feature.merge(meter_reading_shift, on=['building_id', 'timestamp'], how='left')
       
    train_features['lag_value_ratio_'+str(shift_hours)] = (train_feature['lag_value_'+str(shift_hours)]+1)/(train_feature['meter_reading']+1)

100%|██████████| 8/8 [00:25<00:00,  3.17s/it]


In [8]:
train_features.to_pickle('data/preprocessed/train_features_engineered.pkl')
train_features

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,...,lag_value_24,lag_value_168,lag_value_ratio_-168,lag_value_ratio_-24,lag_value_ratio_-2,lag_value_ratio_-1,lag_value_ratio_1,lag_value_ratio_2,lag_value_ratio_24,lag_value_ratio_168
2,2,0,0.0,0.000000,0,Education,5376,91.0,NaN,19.40625,...,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
3,3,0,0.0,0.000000,0,Education,23685,102.0,NaN,19.40625,...,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
5,5,0,0.0,0.000000,0,Education,8000,100.0,NaN,19.40625,...,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
6,6,0,0.0,0.000000,0,Lodging/residential,27926,81.0,NaN,19.40625,...,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
10,10,0,0.0,0.000000,0,Entertainment/public assembly,370773,91.0,NaN,19.40625,...,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20216091,1441,0,8783.0,242.925003,15,Education,30143,51.0,NaN,-25.71875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20216092,1442,0,8783.0,59.400002,15,Public services,99541,93.0,NaN,-25.71875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20216093,1442,2,8783.0,55.624100,15,Public services,99541,93.0,NaN,-25.71875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20216095,1444,0,8783.0,8.750000,15,Entertainment/public assembly,19619,14.0,NaN,-25.71875,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
del train_features, train_feature, meter_reading_shift
gc.collect()

21

In [10]:
test_features = pd.read_pickle('data/preprocessed/local_test_features.pkl')

In [11]:
for shift_hours in tqdm(shifts):
    test_feature = test_features[['building_id', 'timestamp', 'meter_reading']]
    meter_reading_shift = test_features[['building_id', 'timestamp', 'meter_reading']]
    
    meter_reading_shift['timestamp'] += shift_hours
    meter_reading_shift = meter_reading_shift.rename(columns={'meter_reading':'lag_value_'+str(shift_hours)})
    test_feature = test_feature.merge(meter_reading_shift, on=['building_id', 'timestamp'], how='left')
    
    test_features['lag_value_'+str(shift_hours)] = test_feature['lag_value_'+str(shift_hours)]-test_feature['meter_reading']

100%|██████████| 8/8 [00:28<00:00,  3.50s/it]


In [12]:
for shift_hours in tqdm(shifts):
    test_feature = test_features[['building_id', 'timestamp', 'meter_reading']]
    meter_reading_shift = test_features[['building_id', 'timestamp', 'meter_reading']]
    
    meter_reading_shift['timestamp'] += shift_hours
    meter_reading_shift = meter_reading_shift.rename(columns={'meter_reading':'lag_value_'+str(shift_hours)})
    test_feature = test_feature.merge(meter_reading_shift, on=['building_id', 'timestamp'], how='left')
    
    test_features['lag_value_ratio_'+str(shift_hours)] = (test_feature['lag_value_'+str(shift_hours)]+1)/(test_feature['meter_reading']+1)

100%|██████████| 8/8 [00:28<00:00,  3.56s/it]


In [13]:
test_features.to_pickle('data/preprocessed/test_features_engineered.pkl')
test_features

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,...,lag_value_24,lag_value_168,lag_value_ratio_-168,lag_value_ratio_-24,lag_value_ratio_-2,lag_value_ratio_-1,lag_value_ratio_1,lag_value_ratio_2,lag_value_ratio_24,lag_value_ratio_168
0,0,0,0.0,0.000000,0,Education,7432,108.0,NaN,19.40625,...,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
1,1,0,0.0,0.000000,0,Education,2720,104.0,NaN,19.40625,...,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
4,4,0,0.0,0.000000,0,Education,116607,75.0,NaN,19.40625,...,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
7,7,0,0.0,0.000000,0,Education,121074,89.0,NaN,19.40625,...,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
8,8,0,0.0,0.000000,0,Education,60809,103.0,NaN,19.40625,...,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20216090,1440,0,8783.0,154.750000,15,Lodging/residential,150294,87.0,NaN,-25.71875,...,-951.622009,-19.182701,0.820852,1.973404,0.928662,0.654081,0.983061,1.000000,0.067522,0.782466
20216094,1443,0,8783.0,64.949997,15,Education,40311,13.0,NaN,-25.71875,...,-1732.500000,33.158497,0.549223,0.298993,0.806240,6.024014,0.805616,0.785609,0.086008,1.758776
20216096,1445,0,8783.0,4.825000,15,Education,4298,NaN,NaN,-25.71875,...,-3.709007,4.000000,0.810758,0.303830,0.984628,0.512589,0.299153,0.994819,0.952873,1.025478
20216098,1447,0,8783.0,159.574997,15,Lodging/residential,29775,101.0,NaN,-25.71875,...,-69.141998,179.500000,1.735464,1.457574,0.993885,2.553040,1.190874,0.700518,0.154785,2.143312
